# Somo la 04 - Mfumo wa Muundo wa Matumizi ya Zana

Katika somo hili utajifunza muundo wa **Matumizi ya Zana** kwa mawakala wa AI ukitumia Microsoft Agent Framework (Python). Tunashughulikia:

- Kueleza zana za fungsi kwa kivitendo cha `@tool` na vigezo vyenye aina
- Kutoa miundo ya zana ili mfano ujue kila zana inavyofanya kazi gani
- Kudhibiti utekelezaji wa zana kwa `approval_mode`
- Kurudisha **matokeo yaliyopangwa** kupitia mifano ya Pydantic na `response_format`

Hali ni ya **mwakala wa uhifadhi wa safari** ambaye anaweza kutafuta maeneo, kuangalia upatikanaji, na kupata taarifa za ndege.


## Usanidi


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Kufafanua Zana kwa Dekoreta ya @tool

Dekoreta ya `@tool` hubadilisha kazi ya kawaida ya Python kuwa zana ambayo wakala anaweza kuitisha.
Mambo muhimu:

- **docstring** hutumiwa kama maelezo ya zana ambayo mfano unaona.
- **Aina za maelezo** (ikiwemo `Annotated` zenye maelezo) huainisha muundo wa zana.
- `approval_mode` hudhibiti kama mtumiaji lazima aidhinishe kila wito kabla haujatekelezwa.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Kuunda Wakala kwa Zana Nyingi

Pitia zana zote mbili kwa mteja ili mfano uweze kuitumia ile yoyote anayohitaji kujibu swali la mtumiaji.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Matokeo Yaliyoandaliwa na Zana

Kwa kuweka `response_format` kuwa mfano wa Pydantic, wakala tunazuiliwa kurudisha kitu cha JSON chenye aina sahihi badala ya maandishi huru. Hii ni muhimu wakati msimbo unaofuata unahitaji kutumia matokeo kwa njia ya programu.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Mifano ya Idhini ya Zana

Kiparameta `approval_mode` kwenye `@tool` kinasimamia kama wito wa zana unahitaji idhini ya binadamu kabla ya kutekeleza:

| Mode | Tabia |
|---|---|
| `"never_require"` | Zana zinaendesha moja kwa moja — hakuna uthibitisho wa mtumiaji unahitajika. |
| `"always_require"` | Kila wito lazima udhibitishwe na mtumiaji kabla ya kutekelezwa. |

Tumia `"always_require"` kwa zana ambazo zina athari za upande (km. kuweka tiketi ya ndege, kutoza kadi ya mkopo) ili binadamu abaki katika mzunguko.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Muhtasari

Katika somo hili ulijifunza jinsi ya:

1. **Kufafanua zana** ukitumia decorater ya `@tool` pamoja na vigezo vya aina na docstrings zinazotumika kama schema ya zana.
2. **Kuunda zana nyingi** ili wakala aweze kuzitumia kwa mfululizo kujibu maswali tata.
3. **Kurudisha matokeo yenye muundo** kwa kupitisha mfano wa Pydantic kama `response_format`.
4. **Kudhibiti idhini ya zana** kwa kutumia `approval_mode` kuweka binadamu katikati kwa shughuli nyeti.

Mifumo hii ni msingi wa kujenga mawakala wa kuaminika, walio tayari kwa uzalishaji ambao wanaweza kuingiliana na mifumo ya nje kwa usalama.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kiarifa cha kutojibu**:  
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kuhakikisha usahihi, tafadhali fahamu kuwa tafsiri zilizofanywa kwa otomatiki zinaweza kuwa na makosa au upotoshaji. Hati ya asili katika lugha yake ya mama inapaswa kuzingatiwa kama chanzo cha kuheshimika. Kwa taarifa muhimu, tafsiri ya kitaalamu kwa binadamu inapendekezwa. Hatuwezi kuwajibika kwa uelewa au tafsiri potofu zitokanazo na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
